In [0]:
%run "../01_setup/03_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, current_timestamp, col, initcap


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

#Bronze Proessing

In [0]:
base_path = f"s3://child-company-data/{data_source}/*.csv"


In [0]:

df = spark.read.format('csv')\
    .option('header', True)\
    .load(base_path)\
        .withColumn("read_timestamp", lit(current_timestamp()))\
            .select("*", "_metadata.file_name", "_metadata.file_size")


In [0]:
df.write.mode('overwrite').format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

#Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-01-01T04:52:46.041Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404


In [0]:
df_bronze.groupBy('customer_id').count().filter(F.col('count')>1).display()

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
print("Rows before dropping duplicates:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print("Rows after dropping duplicates:", df_silver.count())


Rows before dropping duplicates: 39
Rows after dropping duplicates: 35


In [0]:
df_silver.filter(F.col('customer_name') != F.trim((F.col('customer_name')))).display()

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-01-01T04:52:46.041Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn('customer_name', F.trim(F.col('customer_name')))


In [0]:
df_silver.filter(F.col('customer_name') != F.trim((F.col('customer_name')))).display()

customer_id,customer_name,city,read_timestamp,file_name,file_size


In [0]:
df_silver.select('city').distinct().display()

city
Bengaluru
Hyderabad
New Delhi
Bengalore
Hyderabadd
null
Hyderbad
NewDelhee
NewDelhi
Bengaluruu


In [0]:
city_mapping = {'Hyderabadd': 'Hyderabad',
                'Hyderbad': 'Hyderabad',
                'NewDelhee': 'New Delhi',
                'NewDelhi':'New Delhi',
                'NewDheli': 'New Delhi',
                'Bengaluruu': 'Bengaluru',
                "Bengalore": 'Bengaluru'
                }

allowed = ['Hyderabad', 'Bengaluru', 'New Delhi']

df_silver = df_silver.replace(city_mapping, subset = ['city'])\
    .withColumn("city",
                F.when(F.col('city').isNull(), None)
                .when(F.col('city').isin(allowed), F.col('city'))
                .otherwise(None)
                )
    
df_silver.select('city').distinct().display()

city
Bengaluru
Hyderabad
New Delhi
null


In [0]:
df_silver.select('customer_name').distinct().display()

customer_name
FitFuel Market
Athlete's Choice Store
Endurance Foods
HydroBoost Nutrition
MacroBite Superfoods
MacroBite superfoods
PowerSnack Hub
PowerSnack hub
SprintX nutrition
SprintX Nutrition


In [0]:
df_silver = df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(), None)
    .otherwise(F.initcap('customer_name'))
)


In [0]:
df_silver.select('customer_name').distinct().display()

customer_name
Fitfuel Market
Athlete's Choice Store
Endurance Foods
Hydroboost Nutrition
Macrobite Superfoods
Powersnack Hub
Sprintx Nutrition
Zenathlete Foods
Peak Performance Store
Primefuel Nutrition


In [0]:
df_silver.filter(F.col('city').isNull()).display()

customer_id,customer_name,city,read_timestamp,file_name,file_size
789403,Sprintx Nutrition,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789420,Zenathlete Foods,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789521,Primefuel Nutrition,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789603,Recovery Lane,null,2026-01-01T04:52:46.041Z,customers.csv,1404


In [0]:
null_customer_names = ["Sprintx Nutrition", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]

df_silver.filter(F.col('customer_name').isin(null_customer_names)).display()

customer_id,customer_name,city,read_timestamp,file_name,file_size
789401,Sprintx Nutrition,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404
789402,Sprintx Nutrition,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789403,Sprintx Nutrition,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789420,Zenathlete Foods,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789421,Zenathlete Foods,Hyderabad,2026-01-01T04:52:46.041Z,customers.csv,1404
789422,Zenathlete Foods,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404
789520,Primefuel Nutrition,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404
789521,Primefuel Nutrition,null,2026-01-01T04:52:46.041Z,customers.csv,1404
789522,Primefuel Nutrition,New Delhi,2026-01-01T04:52:46.041Z,customers.csv,1404
789601,Recovery Lane,Bengaluru,2026-01-01T04:52:46.041Z,customers.csv,1404


In [0]:
customer_city_fix = {'789403': 'New Delhi',
                     '789420': 'Bengaluru',
                     '789521': 'Hyderabad',
                     '789603': 'Hyderabad'
                     }

df_fix = spark.createDataFrame([(k, v) for k, v in customer_city_fix.items()], ['customer_id', 'fixed_city'])


In [0]:
df_silver = df_silver.join(df_fix, on = 'customer_id', how = 'left')\
    .withColumn('city', F.coalesce('city', 'fixed_city'))\
        .drop('fixed_city')



In [0]:
df_silver = df_silver.withColumn('customer', F.concat_ws("-", 'customer_name', F.coalesce(F.col('city'), F.lit('Unknown'))))\
    .withColumn("market", F.lit("India"))\
        .withColumn("platform", F.lit("Sports Bar"))\
            .withColumn("channel", F.lit("Acquisition"))

In [0]:
df_silver.write.format('delta')\
    .mode('overwrite')\
        .option('delta.enableChangeDataFeed', 'true')\
            .option('mergeSchema', True)\
            .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

#Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

In [0]:
df_gold.write\
    .format('delta')\
        .option('delta.enableChangeDataFeed', 'true')\
                .mode('overwrite')\
                    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
from delta.tables import DeltaTable


delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'channel'
)

In [0]:
delta_table.alias('target').merge(
    source = df_child_customers.alias('source'),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]